# 03 · Model Architecture

EfficientNet-B0 + 4-output regression head. We inspect parameter
counts, the freezing strategy across training phases, and verify
the forward pass shape on a dummy batch.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch
from src.models.astrolocnet import AstroLocNet, freeze_backbone, unfreeze_last_n_blocks

model = AstroLocNet(pretrained=False)
total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.head_parameters)
backbone = total - head
print(f'Total params:    {total:>10,}')
print(f'Backbone params: {backbone:>10,} ({backbone/total:.1%})')
print(f'Head params:     {head:>10,} ({head/total:.1%})')

## Forward pass sanity check

In [ ]:
x = torch.randn(4, 3, 224, 224)
y = model(x)
print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(y.shape), '<- (B, [ra, dec, rotation, log_scale])')

## Freezing strategy across phases

In [ ]:
def count_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

snapshots = {}
snapshots['Phase 0 (all unfrozen)'] = count_trainable(model)
freeze_backbone(model)
snapshots['Phase 1 (head only)'] = count_trainable(model)
unfreeze_last_n_blocks(model, n=3)
snapshots['Phase 2 (last 3 blocks + head)'] = count_trainable(model)

import numpy as np
fig, ax = plt.subplots(figsize=(8, 4))
names, counts = zip(*snapshots.items())
ax.barh(names, [c / 1e6 for c in counts], color='#7c5cff')
ax.set_xlabel('Trainable params (millions)')
ax.set_title('Trainable parameters per training phase')
for i, c in enumerate(counts):
    ax.text(c / 1e6, i, f'  {c:,}', va='center')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/03_trainable_per_phase.png', dpi=150)
plt.show()

## Block-wise freezing map (which features train when?)

In [ ]:
import numpy as np
# Reset and build per-block training mask for each phase.
phases = ['phase1', 'phase2', 'phase3']
matrix = np.zeros((len(phases), len(model.backbone.features)), dtype=bool)
for row, phase in enumerate(phases):
    freeze_backbone(model)
    if phase != 'phase1':
        unfreeze_last_n_blocks(model, n=3)
    for idx, block in enumerate(model.backbone.features):
        matrix[row, idx] = any(p.requires_grad for p in block.parameters())
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(matrix.astype(float), aspect='auto', cmap='Purples')
ax.set_yticks(range(len(phases))); ax.set_yticklabels(phases)
ax.set_xticks(range(len(model.backbone.features)))
ax.set_xticklabels([f'block_{i}' for i in range(len(model.backbone.features))], rotation=45)
ax.set_title('Which backbone blocks train per phase (purple = trainable)')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/03_freezing_map.png', dpi=150)
plt.show()